In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.paths import MANUAL_PATH, PROCESSED_PATH
from src.portfolio import (
    compute_avg_cost_simple,
    compute_unrealized_pnl,
    classify_rebalance_action,
    allocate_buy_only_suggestion,
)

MANUAL_PATH.mkdir(parents=True, exist_ok=True)

## 1. Setup and Load Transactions

In [ ]:
DATA_PATH = PROCESSED_PATH / "transactions_clean.csv"

df = pd.read_csv(DATA_PATH)

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

df.head()

## 2. Prepare Transaction Data

In [ ]:
trading = df[
    df["category"] == "TRADING"
].copy()

trading["month"] = trading["date"].dt.to_period("M")

trading[[
    "date",
    "type",
    "name",
    "symbol",
    "asset_class",
    "shares",
    "price",
    "amount",
    "fee",
    "tax"
]].head(10)

In [ ]:
buys = trading[
    trading["type"] == "BUY"
].copy()

sells = trading[
    trading["type"] == "SELL"
].copy()

trading["buy_amount"] = np.where(
    trading["type"] == "BUY",
    trading["amount"].abs(),
    0
)

trading["sell_amount"] = np.where(
    trading["type"] == "SELL",
    trading["amount"],
    0
)

trading["fee_paid"] = trading["fee"].fillna(0).abs()
trading["tax_paid"] = trading["tax"].fillna(0).abs()

In [ ]:
position_events = df[
    (
        (df["category"] == "TRADING")
        | (df["economic_type"] == "Corporate Action")
    )
    & df["shares"].notna()
].copy()

position_events[[
    "date",
    "category",
    "type",
    "name",
    "symbol",
    "asset_class",
    "shares",
    "amount"
]].sort_values("date").head()

## 3. Holdings Analysis

In [ ]:
share_positions = (
    position_events
    .groupby(["symbol", "name", "asset_class"], as_index=False)
    .agg(
        shares=("shares", "sum"),
        n_position_events=("shares", "size"),
        first_position_event=("date", "min"),
        last_position_event=("date", "max")
    )
)

trade_stats = (
    trading
    .groupby(["symbol", "name", "asset_class"], as_index=False)
    .agg(
        total_bought=("buy_amount", "sum"),
        total_sold=("sell_amount", "sum"),
        fees_paid=("fee_paid", "sum"),
        taxes_paid=("tax_paid", "sum"),
        n_buys=("type", lambda x: (x == "BUY").sum()),
        n_sells=("type", lambda x: (x == "SELL").sum()),
        first_trade_date=("date", "min"),
        last_trade_date=("date", "max")
    )
)

holdings_all = share_positions.merge(
    trade_stats,
    on=["symbol", "name", "asset_class"],
    how="left"
)

fill_zero_cols = [
    "total_bought",
    "total_sold",
    "fees_paid",
    "taxes_paid",
    "n_buys",
    "n_sells"
]

holdings_all[fill_zero_cols] = holdings_all[fill_zero_cols].fillna(0)

holdings_all["avg_cost_simple"] = compute_avg_cost_simple(
    holdings_all["total_bought"],
    holdings_all["shares"]
)

holdings_all = holdings_all.sort_values(
    "total_bought",
    ascending=False
)

holdings_all

In [ ]:
current_holdings = holdings_all[
    holdings_all["shares"] > 1e-8
].copy()

closed_positions = holdings_all[
    holdings_all["shares"].abs() <= 1e-8
].copy()

problem_positions = holdings_all[
    holdings_all["shares"] < -1e-8
].copy()


In [ ]:
current_holdings

## 4. Ticker Mapping

In [ ]:
assets_to_map = (
    current_holdings[["symbol", "name", "asset_class"]]
    .drop_duplicates()
    .sort_values(["asset_class", "name"])
    .copy()
)

assets_to_map["yf_ticker"] = ""
assets_to_map["price_currency"] = "EUR"
assets_to_map["mapping_notes"] = ""

assets_to_map.to_csv(
    MANUAL_PATH / "ticker_map_template.csv",
    index=False
)

assets_to_map

In [ ]:
TICKER_MAP_PATH = MANUAL_PATH / "ticker_map_manual.csv"

# Generic example mapping — used only to seed ticker_map_manual.csv on a
# fresh checkout (that file is gitignored, see data/manual/*, and holds the
# real symbol -> yf_ticker/currency mapping). Edit the CSV directly (or fill
# in ticker_map_template.csv above) to map your own holdings; this cell
# intentionally contains no real ISIN/holding data, since it is published
# as part of the notebook source.
ticker_map_example_seed = pd.DataFrame([
    {"symbol": "IE00EXAMPLE01", "yf_ticker": "VWCE.DE", "price_currency": "EUR"},
    {"symbol": "IE00EXAMPLE02", "yf_ticker": "IEMA.AS", "price_currency": "EUR"},
    {"symbol": "IE00EXAMPLE03", "yf_ticker": "4GLD.DE", "price_currency": "EUR"},
    {"symbol": "US0EXAMPLE04", "yf_ticker": "AAPL", "price_currency": "USD"},
    {"symbol": "US0EXAMPLE05", "yf_ticker": "MSFT", "price_currency": "USD"},
    {"symbol": "XXEXAMPLE0006", "yf_ticker": "BTC-EUR", "price_currency": "EUR"},
])

In [ ]:
if not TICKER_MAP_PATH.exists():
    ticker_map_example_seed.to_csv(TICKER_MAP_PATH, index=False)
    print(f"Seeded example ticker map at {TICKER_MAP_PATH}")
else:
    print(f"Using existing ticker map at {TICKER_MAP_PATH} (not overwritten)")

ticker_map_df = pd.read_csv(TICKER_MAP_PATH)

ticker_map = ticker_map_df.set_index("symbol")["yf_ticker"].to_dict()
currency_map = ticker_map_df.set_index("symbol")["price_currency"].to_dict()

In [ ]:
current_holdings["yf_ticker"] = current_holdings["symbol"].map(ticker_map)
current_holdings["price_currency"] = current_holdings["symbol"].map(currency_map)

current_holdings[current_holdings["yf_ticker"].isna()]

In [ ]:
trackable_holdings = current_holdings[
    current_holdings["yf_ticker"].notna()
].copy()

unmapped_holdings = current_holdings[
    current_holdings["yf_ticker"].isna()
].copy()

In [ ]:
# Holdings without a Yahoo ticker can't be valued. On a first run with your own
# export that is every holding: fill in ticker_map_template.csv (written above)
# and save it as ticker_map_manual.csv. Holdings you deliberately leave unmapped
# (e.g. a delisted stock) are listed here and left out of the valuation.
if not unmapped_holdings.empty:
    unmapped_list = "\n".join(
        f"  {row.symbol}  {row['name']}" for _, row in unmapped_holdings.iterrows()
    )
    message = (
        f"{len(unmapped_holdings)} holding(s) without a Yahoo Finance ticker:\n{unmapped_list}\n"
        f"Fill in yf_ticker and price_currency for them in {MANUAL_PATH / 'ticker_map_template.csv'} "
        f"and save the file as {TICKER_MAP_PATH.name} (see README, 'Use It with Your Own Data')."
    )
    if trackable_holdings.empty:
        raise RuntimeError(message)
    print(
        f"Not valued (no Yahoo Finance ticker):\n{unmapped_list}\n"
        f"Fine if deliberate (e.g. delisted); otherwise add them to {TICKER_MAP_PATH.name}."
    )


In [ ]:
trackable_holdings

## 5. Load current prices

In [ ]:
import yfinance as yf

In [ ]:
price_data = []

for ticker in trackable_holdings["yf_ticker"].dropna().unique():
    hist = yf.download(
        ticker,
        period="5d",
        interval="1d",
        auto_adjust=True,
        progress=False
    )
    
    if not hist.empty:
        close = hist["Close"]
        
        if isinstance(close, pd.DataFrame):
            close = close.iloc[:, 0]
        
        last_price = close.dropna().iloc[-1]
        
        price_data.append({
            "yf_ticker": ticker,
            "current_price_raw": float(last_price)
        })

prices = pd.DataFrame(price_data, columns=["yf_ticker", "current_price_raw"])

# An empty result almost always means no network (e.g. a scheduled run right
# after wake-up) rather than bad tickers — fail with a clear message instead
# of an opaque KeyError in the merge below.
if prices.empty:
    raise RuntimeError(
        "No prices downloaded from yfinance for any holding — check the network "
        "connection (or whether yfinance/Yahoo is down), then re-run."
    )

missing_prices = sorted(
    set(trackable_holdings["yf_ticker"].dropna()) - set(prices["yf_ticker"])
)
if missing_prices:
    print(f"WARNING: no price data for {missing_prices} — these holdings get no market value.")

prices

In [ ]:
trackable_holdings = trackable_holdings.merge(
    prices,
    on="yf_ticker",
    how="left"
)

trackable_holdings

In [ ]:
eurusd = yf.download(
    "EURUSD=X",
    period="5d",
    interval="1d",
    auto_adjust=True,
    progress=False
)

eurusd_close = eurusd["Close"]

if isinstance(eurusd_close, pd.DataFrame):
    eurusd_close = eurusd_close.iloc[:, 0]

eur_usd_rate = float(eurusd_close.dropna().iloc[-1])

usd_to_eur = 1 / eur_usd_rate

usd_to_eur

In [ ]:
trackable_holdings["current_price_eur"] = np.where(
    trackable_holdings["price_currency"] == "USD",
    trackable_holdings["current_price_raw"] * usd_to_eur,
    trackable_holdings["current_price_raw"]
)
trackable_holdings

## 6. Market Value Calculation

In [ ]:
trackable_holdings["market_value"] = trackable_holdings["shares"] * trackable_holdings["current_price_eur"]

total_portfolio_value = trackable_holdings["market_value"].sum()

trackable_holdings["actual_weight"] = trackable_holdings["market_value"] / total_portfolio_value
trackable_holdings = trackable_holdings.sort_values("market_value", ascending=False)

trackable_holdings

In [ ]:
total_portfolio_value

## 7. Unrealized P&L Calculation

In [ ]:
trackable_holdings["cost_basis_simple"] = trackable_holdings["shares"] * trackable_holdings["avg_cost_simple"]

trackable_holdings["unrealized_pnl_simple"], trackable_holdings["unrealized_pnl_pct_simple"] = compute_unrealized_pnl(
    trackable_holdings["market_value"],
    trackable_holdings["cost_basis_simple"]
)

trackable_holdings["pnl_is_approximate"] = trackable_holdings["n_sells"] > 0

df_holdings = trackable_holdings[[
    "symbol",
    "name",
    "asset_class",
    "shares",
    "market_value",
    "cost_basis_simple",
    "unrealized_pnl_simple",
    "unrealized_pnl_pct_simple",
    "actual_weight",
    "pnl_is_approximate"
]].sort_values("market_value", ascending=False)
df_holdings

The unrealized P&L is calculated using a simple average-cost approximation. Exact realized and unrealized gains require a lot-level reconstruction, which can be added later.

## 8. Visualization

In [ ]:
top_holdings = trackable_holdings.sort_values("actual_weight", ascending=False)

plt.figure(figsize=(12,6))
plt.bar(
    top_holdings["name"],
    top_holdings["actual_weight"] * 100
)

plt.title("Current Portfolio Weights")
plt.xlabel("Asset")
plt.ylabel("Weight (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
asset_class_exposure = (
    trackable_holdings
    .groupby("asset_class")["market_value"]
    .sum()
    .sort_values(ascending=False)
)

asset_class_exposure_pct = asset_class_exposure / asset_class_exposure.sum()

asset_class_exposure_pct

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(
    asset_class_exposure_pct.index,
    asset_class_exposure_pct.values * 100
)

plt.title("Portfolio Exposure by Asset Class")
plt.xlabel("Asset Class")
plt.ylabel("Weight (%)")
plt.tight_layout()
plt.show()

## 9. Actual vs Target Weights

In [ ]:
TARGET_WEIGHTS_PATH = MANUAL_PATH / "target_weights_manual.csv"

# Generic example allocation — used only to seed target_weights_manual.csv on a
# fresh checkout (that file is gitignored, see data/manual/*, and holds the
# real, curated target allocation). Edit the CSV directly to change your
# actual targets; this cell intentionally contains no real allocation data,
# since it is published as part of the notebook source.
target_weights_example_seed = pd.DataFrame([
    {"symbol": "IE00EXAMPLE01", "target_weight": 0.65, "note": "example: global equity (VWCE.DE in the example ticker map)"},
    {"symbol": "IE00EXAMPLE02", "target_weight": 0.18, "note": "example: emerging markets (IEMA.AS)"},
    {"symbol": "IE00EXAMPLE03", "target_weight": 0.09, "note": "example: gold (4GLD.DE)"},
    {"symbol": "US0EXAMPLE04", "target_weight": 0.03, "note": "example: single stock (AAPL)"},
    {"symbol": "US0EXAMPLE05", "target_weight": 0.03, "note": "example: single stock (MSFT)"},
    {"symbol": "XXEXAMPLE0006", "target_weight": 0.02, "note": "example: crypto (BTC-EUR)"},
])

In [ ]:
if not TARGET_WEIGHTS_PATH.exists():
    target_weights_example_seed.to_csv(TARGET_WEIGHTS_PATH, index=False)
    print(f"Seeded example target allocation at {TARGET_WEIGHTS_PATH}")
else:
    print(f"Using existing target allocation at {TARGET_WEIGHTS_PATH} (not overwritten)")

target_weights = pd.read_csv(TARGET_WEIGHTS_PATH)[["symbol", "target_weight"]]

target_weights

In [ ]:
weight_comparison = trackable_holdings[[
    "symbol",
    "name",
    "asset_class",
    "market_value",
    "actual_weight"
]].merge(
    target_weights,
    on="symbol",
    how="left"
)

weight_comparison["target_weight"] = weight_comparison["target_weight"].fillna(0)

weight_comparison

In [ ]:
target_sum = target_weights["target_weight"].sum()
assert np.isclose(target_sum, 1.0), f"Target weights sum to {target_sum:.4f}, not 1.0"

In [ ]:
weight_comparison["target_value"] = (
    weight_comparison["target_weight"]
    * total_portfolio_value
)

weight_comparison["value_drift"] = (
    weight_comparison["market_value"]
    - weight_comparison["target_value"]
)

weight_comparison["weight_drift"] = (
    weight_comparison["actual_weight"]
    - weight_comparison["target_weight"]
)

weight_comparison["weight_drift_pp"] = (
    weight_comparison["weight_drift"] * 100
)

weight_comparison = weight_comparison.sort_values(
    "weight_drift",
    ascending=False
)

weight_comparison

In [ ]:
DRIFT_THRESHOLD_PP = 2.0

weight_comparison["rebalance_flag"] = (
    weight_comparison["weight_drift_pp"].abs()
    > DRIFT_THRESHOLD_PP
)

weight_comparison["rebalance_action"] = weight_comparison["weight_drift_pp"].apply(
    lambda drift_pp: classify_rebalance_action(drift_pp, DRIFT_THRESHOLD_PP)
)

weight_comparison

In [ ]:
plot_data = weight_comparison.sort_values("weight_drift_pp")

plt.figure(figsize=(12,6))
plt.barh(
    plot_data["name"],
    plot_data["weight_drift_pp"]
)
plt.axvline(0, linestyle="--")
plt.axvline(DRIFT_THRESHOLD_PP, color="red", linestyle="--", label=f"Drift Threshold (+{DRIFT_THRESHOLD_PP} pp)")
plt.axvline(-DRIFT_THRESHOLD_PP, color="red", linestyle="--", label=f"Drift Threshold (-{DRIFT_THRESHOLD_PP} pp)")
plt.title("Actual vs Target Weight Drift")
plt.xlabel("Drift in percentage points")
plt.ylabel("Asset")
plt.tight_layout()
plt.show()

## 10. Buy-only Rebalancing Suggestion

In [ ]:
NEW_CASH_TO_INVEST = 300  # notebook default; adjustable live in the dashboard

buy_only_suggestion = weight_comparison[
    weight_comparison["rebalance_action"] == "Add / buy more"
].copy()

buy_only_suggestion["underweight_amount"] = -buy_only_suggestion["value_drift"]

buy_only_suggestion["underweight_share"], buy_only_suggestion["suggested_buy_amount"] = allocate_buy_only_suggestion(
    buy_only_suggestion["underweight_amount"],
    NEW_CASH_TO_INVEST
)

buy_only_suggestion = buy_only_suggestion.sort_values(
    "suggested_buy_amount",
    ascending=False
)

buy_only_suggestion[[
    "symbol",
    "name",
    "actual_weight",
    "target_weight",
    "weight_drift_pp",
    "underweight_amount",
    "suggested_buy_amount"
]]

## 11. Save data and KPIs

In [ ]:
portfolio_dashboard = trackable_holdings[[
    "symbol",
    "name",
    "asset_class",
    "shares",
    "yf_ticker",
    "price_currency",
    "current_price_raw",
    "current_price_eur",
    "market_value",
    "actual_weight",
    "cost_basis_simple",
    "unrealized_pnl_simple",
    "unrealized_pnl_pct_simple",
    "pnl_is_approximate"
]].copy()

portfolio_dashboard

In [ ]:
portfolio_kpis = pd.DataFrame([{
    "total_portfolio_value_eur": total_portfolio_value,
    "n_current_holdings_all": len(current_holdings),
    "n_trackable_holdings": len(trackable_holdings),
    "n_unmapped_holdings": len(unmapped_holdings),

    "fund_weight": asset_class_exposure_pct.get("FUND", 0),
    "stock_weight": asset_class_exposure_pct.get("STOCK", 0),
    "crypto_weight": asset_class_exposure_pct.get("CRYPTO", 0),

    "largest_position_name": trackable_holdings.loc[
        trackable_holdings["actual_weight"].idxmax(),
        "name"
    ],
    "largest_position_weight": trackable_holdings["actual_weight"].max(),

    "total_unrealized_pnl_simple": trackable_holdings["unrealized_pnl_simple"].sum(),
    "total_cost_basis_simple": trackable_holdings["cost_basis_simple"].sum(),

    "n_rebalance_flags": weight_comparison["rebalance_flag"].sum(),
    "max_abs_drift_pp": weight_comparison["weight_drift_pp"].abs().max(),

    "usd_to_eur": usd_to_eur
}])

In [ ]:
portfolio_kpis["suggested_buy_amount_total"] = (
    buy_only_suggestion["suggested_buy_amount"].sum()
)

In [ ]:
portfolio_dashboard.to_csv(
    PROCESSED_PATH / "portfolio_dashboard.csv",
    index=False
)

portfolio_kpis.to_csv(
    PROCESSED_PATH / "portfolio_kpis.csv",
    index=False
)

current_holdings.to_csv(
    PROCESSED_PATH / "current_holdings_all.csv",
    index=False
)

trackable_holdings.to_csv(
    PROCESSED_PATH / "current_holdings_trackable.csv",
    index=False
)

unmapped_holdings.to_csv(
    PROCESSED_PATH / "current_holdings_unmapped.csv",
    index=False
)

weight_comparison.to_csv(
    PROCESSED_PATH / "actual_vs_target_weights.csv",
    index=False
)

buy_only_suggestion.to_csv(
    PROCESSED_PATH / "buy_only_rebalancing_suggestion.csv",
    index=False
)

## 12. Sanity Checks

In [ ]:
assert problem_positions.empty
assert trackable_holdings["current_price_eur"].notna().all()
assert np.isclose(trackable_holdings["actual_weight"].sum(), 1.0)
assert np.isclose(target_weights["target_weight"].sum(), 1.0), (
    f"Target weights in {TARGET_WEIGHTS_PATH} sum to {target_weights['target_weight'].sum():.3f}, not 1"
)

missing_target_weights = set(trackable_holdings["symbol"]) - set(target_weights["symbol"])
assert len(missing_target_weights) == 0, (
    f"Missing target weights for: {missing_target_weights} -- add them to "
    f"{TARGET_WEIGHTS_PATH} (weights must sum to 1)"
)

assert weight_comparison["target_weight"].notna().all()
assert np.isclose(weight_comparison["target_weight"].sum(), 1.0)